# AgingClockBench — Advanced Benchmarking

This notebook covers:
1. **Stratified analysis** — benchmark by age group and sex
2. **KDM custom fit** — derive reference parameters from your own cohort
3. **Cox PH deep-dive** — compare clocks as mortality predictors
4. **Kaplan-Meier plots** — survival by acceleration quartile

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
%matplotlib inline

from agingclockbench import PhenoAge, KDM, BenchmarkSuite
from agingclockbench.datasets import load_nhanes_sample

df = load_nhanes_sample()
print(f"NHANES sample: {len(df)} participants, {df.mortstat.sum()} deaths")

## 1. Full benchmark on NHANES with mortality

In [ ]:
pa = PhenoAge().transform(df)
kdm = KDM().transform(df.drop(columns=['crp_mg_l']))

suite = BenchmarkSuite(mortality_col='mortstat', followup_col='permth_exm')
report = suite.run(df, {'PhenoAge': pa, 'KDM': kdm})
report.to_dataframe()

## 2. Kaplan-Meier survival curves

In [ ]:
fig = report.plot_km_survival(n_quartiles=4)
fig.savefig('km_survival.png', dpi=120, bbox_inches='tight')

## 3. Stratified analysis by sex

In [ ]:
rows = []
for sex in ['male', 'female']:
    sub = df[df.sex == sex].reset_index(drop=True)
    pa_sub = PhenoAge().transform(sub)
    suite_sub = BenchmarkSuite(mortality_col='mortstat', followup_col='permth_exm')
    rep_sub = suite_sub.run(sub, {'PhenoAge': pa_sub})
    row = rep_sub.to_dataframe().iloc[0].to_dict()
    row['Sex'] = sex
    row['N'] = len(sub)
    rows.append(row)

pd.DataFrame(rows)[['Sex', 'N', 'Pearson r', 'Mort HR (per SD accel)', 'Mort p-value']]

## 4. Stratified analysis by age group

In [ ]:
rows = []
for label, lo, hi in [('<50', 20, 49), ('50-64', 50, 64), ('65+', 65, 90)]:
    sub = df[(df.age >= lo) & (df.age <= hi)].reset_index(drop=True)
    if len(sub) < 50:
        continue
    pa_sub = PhenoAge().transform(sub)
    suite_sub = BenchmarkSuite(mortality_col='mortstat', followup_col='permth_exm')
    rep_sub = suite_sub.run(sub, {'PhenoAge': pa_sub})
    row = rep_sub.to_dataframe().iloc[0].to_dict()
    row['Age group'] = label
    row['N'] = len(sub)
    rows.append(row)

pd.DataFrame(rows)[['Age group', 'N', 'Pearson r', 'Mort HR (per SD accel)', 'Mort p-value']]

## 5. KDM with custom reference parameters

In [ ]:
# Split data: use first half to fit, second half to evaluate
kdm_df = df.drop(columns=['crp_mg_l'])
train = kdm_df.iloc[:2000].reset_index(drop=True)
test  = kdm_df.iloc[2000:].reset_index(drop=True)

# Fit on training set
clock_fitted = KDM().fit(train)
print(f"Fitted s_BA: {clock_fitted._s_ba:.2f} years")

# Evaluate on test set
result_fitted = clock_fitted.transform(test)
result_default = KDM().transform(test)

print(f"Custom fit  — Pearson r with age: {result_fitted.biological_ages.corr(test.age):.4f}")
print(f"Default fit — Pearson r with age: {result_default.biological_ages.corr(test.age):.4f}")

## 6. Export full interactive report

In [ ]:
report.to_html('nhanes_benchmark_report.html')
print('Open nhanes_benchmark_report.html in your browser for the full interactive report.')